In [6]:
import numpy as np

class AdaptiveKuramotoSimulator:
    """
    Co-Evolutionary Adaptive Kuramoto Model Simulator
    """
    def __init__(self, N=100, alpha=0.1, nu=0.6, theta_c_deg=35.0):
        self.N = N
        self.alpha = alpha
        self.nu = nu
        self.cos_theta_c = np.cos(np.radians(theta_c_deg))
        
        self.theta = np.random.uniform(0, 2 * np.pi, N)
        self.omega = np.random.normal(0, 1.0, N)
        self.W = np.random.uniform(0.0, 0.05, (N, N))
        np.fill_diagonal(self.W, 0.0)

    def step(self, dt=0.01):
        diffs = self.theta[None, :] - self.theta[:, None]
        interaction = np.sum(self.W * np.sin(diffs), axis=1)
        d_theta = self.omega + interaction / self.N
        
        cos_diffs = np.cos(diffs)
        heaviside = (cos_diffs > self.cos_theta_c).astype(float)
        dW = -self.alpha * self.W + self.nu * heaviside * self.W
        
        self.theta = (self.theta + dt * d_theta) % (2 * np.pi)
        self.W = np.clip(self.W + dt * dW, 0.0, 4.0)
        np.fill_diagonal(self.W, 0.0)

    def run_experiment(self, total_time=40.0, dt=0.01):
        steps = int(total_time / dt)
        t_history = []
        r_history = []
        w_history = []

        for s in range(steps):
            self.step(dt)
            if s % 50 == 0:
                t = s * dt
                z = np.mean(np.exp(1j * self.theta))
                r = np.abs(z)
                mean_w = np.mean(self.W)
                t_history.append(t)
                r_history.append(r)
                w_history.append(mean_w)
                
        return np.array(t_history), np.array(r_history), np.array(w_history)


def analyze_functional_parameters(sweep_results):
    """
    Tarama sonuçlarını analiz ederek sistemin başarılı olduğu işlevsel 
    çalışma penceresini (functional window) ve optimal parametreleri çıkarır.
    """
    # Global senkronizasyon sağlayan (r > 0.6) başarılı testleri filtrele
    successful_runs = [r for r in sweep_results if r["steady_r"] > 0.6]
    
    print("\n" + "="*50)
    print(" FUNCTIONAL PARAMETER EXTRACTION REPORT")
    print("="*50)
    
    if not successful_runs:
        print("Status: FAILED")
        print("Conclusion: No functional parameter window found for global synchronization under current settings.")
        return None

    theta_values = [r["theta_c_deg"] for r in successful_runs]
    min_theta = min(theta_values)
    max_theta = max(theta_values)
    
    # En yüksek sipariş parametresine (r) sahip optimal konfigürasyonu bul
    optimal_run = max(successful_runs, key=lambda x: x["steady_r"])
    
    print(f"Status: SUCCESS")
    print(f"Functional Window (theta_c range): [{min_theta}° - {max_theta}°]")
    print(f"Optimal Phase Threshold (theta_c): {optimal_run['theta_c_deg']}°")
    print(f"  -> Achieved Steady-State r:      {optimal_run['steady_r']:.4f}")
    print(f"  -> Corresponding Mean Weight <W>: {optimal_run['steady_w']:.4f}")
    print("="*50)
    
    return {
        "functional_window": (min_theta, max_theta),
        "optimal_theta_c": optimal_run["theta_c_deg"],
        "optimal_r": optimal_run["steady_r"]
    }


if __name__ == "__main__":
    print("="*70)
    print(" ADAPTIVE KURAMOTO PARAMETER SWEEP & ANALYSIS PIPELINE")
    print("="*70)
    print(f"{'Step':<6} | {'Theta_c (deg)':<15} | {'Steady-State r':<16} | {'Mean Weight <W>':<16} | {'Regime'}")
    print("-" * 70)

    sweep_results = []

    for i in range(1, 72):
        theta_c = i * 5
        simulator = AdaptiveKuramotoSimulator(N=100, alpha=0.1, nu=0.6, theta_c_deg=theta_c)
        t_arr, r_arr, w_arr = simulator.run_experiment(total_time=40.0, dt=0.01)
        
        steady_r = np.mean(r_arr[-int(len(r_arr) * 0.2):])
        steady_w = np.mean(w_arr[-int(len(w_arr) * 0.2):])
        
        if steady_r < 0.2:
            regime = "Incoherent"
        elif steady_r < 0.6:
            regime = "Partial Sync"
        else:
            regime = "Global Sync"

        print(f"{i:<6} | {theta_c:<15.1f} | {steady_r:<16.4f} | {steady_w:<16.4f} | {regime}")
        
        sweep_results.append({
            "step": i,
            "theta_c_deg": theta_c,
            "steady_r": steady_r,
            "steady_w": steady_w,
            "regime": regime
        })

    print("-" * 70)
    
    # Analiz fonksiyonunu çağırarak sonuçları rapora dönüştür
    analysis_report = analyze_functional_parameters(sweep_results)

 ADAPTIVE KURAMOTO PARAMETER SWEEP & ANALYSIS PIPELINE
Step   | Theta_c (deg)   | Steady-State r   | Mean Weight <W>  | Regime
----------------------------------------------------------------------
1      | 5.0             | 0.0631           | 0.0023           | Incoherent
2      | 10.0            | 0.0989           | 0.0094           | Incoherent
3      | 15.0            | 0.1227           | 0.0112           | Incoherent
4      | 20.0            | 0.0857           | 0.0269           | Incoherent
5      | 25.0            | 0.1383           | 0.0469           | Incoherent
6      | 30.0            | 0.1060           | 0.0579           | Incoherent
7      | 35.0            | 0.1092           | 0.1221           | Incoherent
8      | 40.0            | 0.1962           | 0.1731           | Incoherent
9      | 45.0            | 0.1493           | 0.2325           | Incoherent
10     | 50.0            | 0.1413           | 0.3896           | Incoherent
11     | 55.0            | 0.2380         